# 09 — Foundry IQ with Web IQ only

Create a separate knowledge base with exactly one source: basic Web IQ web search. Calls go directly to Azure AI Search, without APIM, Blob, SharePoint, indexes or agents. Notebooks 11 and 12 are not prerequisites.

The source stores the Web IQ API key, so both notebook and Foundry IQ portal queries can authenticate. Retrieval output includes MCP activity, reference URLs and retrieved passages, not just the generated answer.

## Required permissions

| Function | Required access |
|---|---|
| Create/update/delete the knowledge source and knowledge base | Search Service Contributor on the existing Search service |
| Query the knowledge base | Search Index Data Reader, or Search Index Data Contributor, on Search |
| Search calls the existing chat deployment | Search's managed identity needs Cognitive Services User, or equivalent inference access, on the parent Foundry resource |
| Web IQ searches | A valid Web IQ profile API key in `WEBIQ_API_KEY` |

`WORKSHOP_PREFIX` controls the names of this notebook's separate resources. It does not enforce access isolation. Source-authoring access must be restricted: authorized queries consume the stored key's shared Web IQ quota and billing.

Progression: this Web-only lab is followed by [10 - Web IQ + Blob](../python-notebooks/10-foundry-iq-webiq-blob.ipynb), then [11 - Web IQ + Blob + SharePoint](../python-notebooks/11-foundry-iq-knowledge-base.ipynb).
The existing `WORKSHOP_PREFIX-nb14-webiq` and `WORKSHOP_PREFIX-nb14-kb` names are deliberately retained from this lab's former number so reruns update the same Azure definitions.


Set the kernel working directory to `c-sharp-notebooks`. Configuration loads `..\.env` first, then `..\.env.azure`, both with `override=True`; set `WORKSHOP_PREFIX` before running (no prefix prompt).

Install .NET 10, Azure CLI and Polyglot Notebooks with a C#/.NET Interactive kernel. This notebook is standalone: all helpers and REST payloads are inline; no console app or earlier lab is imported. Use the existing Search service, chat deployment and Web IQ profile; do not deploy new infrastructure.


## 1. Configuration and sign-in

Set the kernel working directory to `c-sharp-notebooks`. The parent `.env` loads first,
then `.env.azure`, overriding existing environment values; `.env.azure` wins.
Required: `WORKSHOP_PREFIX` (3–20 lowercase letters/digits starting with a letter; no prompt),
`AZURE_TENANT_ID` (GUID), `AZURE_SEARCH_ENDPOINT`, `AZURE_AI_ACCOUNT_NAME`,
`AZURE_AI_MODEL_DEPLOYMENT_NAME`, `WEBIQ_API_KEY`. `AZURE_AI_MODEL_NAME` defaults to `gpt-5-mini`.
See [settings](../.env.example). Use a public-cloud HTTPS Search service root.

Run `az login --tenant <AZURE_TENANT_ID>` in a terminal first. The inline BCL helper uses
the same Azure CLI Search credential scope `https://search.azure.com/.default` as Python09,
with a 60-second process timeout and disposal of the CLI process/token acquisition resources.
It does not print bearer tokens, CLI stderr or the Web IQ key. A CLI timeout occurs before
Search/Web IQ; it is not proof of an invalid Web IQ key. Test expiry only, not token output:

```powershell
az account get-access-token --tenant '<tenant-id>' --resource https://search.azure.com --query expiresOn --output tsv
```

All requests use REST `2026-08-01-preview`. There are no automatic paid retries or caches.
Do not display the source payload/settings key or save sensitive output in notebook commits.


In [ ]:
using System;
using System.IO;
using System.Linq;
using System.Collections.Generic;
using System.Net.Http;
using System.Threading;
using System.Threading.Tasks;
using System.Diagnostics;
using System.Net.Http.Headers;
using System.Text;
using System.Text.Json;
using System.Text.Json.Nodes;
using System.Text.RegularExpressions;

public sealed class WebIqSettings(string tenant, string searchEndpoint, string account,
    string deployment, string model, string prefix, string webIqKey)
{
    public string Tenant { get; } = tenant;
    public string SearchEndpoint { get; } = searchEndpoint;
    public string Account { get; } = account;
    public string Deployment { get; } = deployment;
    public string Model { get; } = model;
    public string Prefix { get; } = prefix;
    public string WebIqKey { get; } = webIqKey;
    public string ModelEndpoint => $"https://{Account}.openai.azure.com";
    public string SourceName => Prefix + "-nb14-webiq";
    public string BaseName => Prefix + "-nb14-kb";

    public static WebIqSettings Load(string root)
    {
        LoadEnvFile(Path.Combine(root, ".env"));
        LoadEnvFile(Path.Combine(root, ".env.azure"));
        string Required(string name) => Environment.GetEnvironmentVariable(name)?.Trim() is { Length: > 0 } value
            ? value : throw new InvalidOperationException("Missing setting: " + name + ". See .env.example.");
        var settings = new WebIqSettings(Required("AZURE_TENANT_ID"), Required("AZURE_SEARCH_ENDPOINT").TrimEnd('/'),
            Required("AZURE_AI_ACCOUNT_NAME"), Required("AZURE_AI_MODEL_DEPLOYMENT_NAME"),
            Environment.GetEnvironmentVariable("AZURE_AI_MODEL_NAME") ?? "gpt-5-mini",
            Required("WORKSHOP_PREFIX"), Required("WEBIQ_API_KEY"));
        settings.Validate();
        return settings;
    }

    public void Validate()
    {
        if (!Guid.TryParseExact(Tenant, "D", out _) || !Regex.IsMatch(Prefix, @"^[a-z][a-z0-9]{2,19}$"))
            throw new InvalidOperationException("Use a tenant GUID and 3-20 lowercase letters/digits prefix starting with a letter.");
        if (!Uri.TryCreate(SearchEndpoint, UriKind.Absolute, out var endpoint) || endpoint.Scheme != "https" ||
            endpoint.UserInfo != "" || !endpoint.IsDefaultPort || endpoint.AbsolutePath != "/" ||
            endpoint.Query != "" || endpoint.Fragment != "" ||
            !Regex.IsMatch(endpoint.Host, @"^[a-z0-9][a-z0-9-]*\.search\.windows\.net$", RegexOptions.IgnoreCase))
            throw new InvalidOperationException("Use a public Azure HTTPS Search service root without credentials, path or query.");
        if (!Regex.IsMatch(Account, @"^[a-zA-Z0-9][a-zA-Z0-9-]*$") ||
            string.IsNullOrWhiteSpace(Deployment) || string.IsNullOrWhiteSpace(Model) || string.IsNullOrWhiteSpace(WebIqKey))
            throw new InvalidOperationException("Use a Foundry account name, chat deployment/model and non-empty WEBIQ_API_KEY.");
    }

    public static void LoadEnvFile(string path)
    {
        if (!File.Exists(path)) return;
        foreach (string line in File.ReadAllLines(path))
        {
            string setting = line.Trim();
            if (setting.Length == 0 || setting.StartsWith('#')) continue;
            if (setting.StartsWith("export ", StringComparison.Ordinal)) setting = setting[7..].TrimStart();
            int equals = setting.IndexOf('=');
            if (equals <= 0) continue;
            string name = setting[..equals].Trim();
            string value = setting[(equals + 1)..].Trim();
            if (value.Length >= 2 && ((value[0] == '"' && value[^1] == '"') || (value[0] == '\'' && value[^1] == '\'')))
                value = value[1..^1];
            else
            {
                int comment = value.IndexOf(" #", StringComparison.Ordinal);
                if (comment >= 0) value = value[..comment].TrimEnd();
            }
            Environment.SetEnvironmentVariable(name, value);
        }
    }
}

public sealed class WebIqTransport(HttpClient client, string tenant, Func<Task<string>>? tokenProvider = null)
{
    public const string Scope = "https://search.azure.com/.default";
    public const string ApiVersion = "2026-08-01-preview";
    readonly Func<Task<string>> getToken = tokenProvider ?? (() => AzureCliTokenAsync(tenant));

    public async Task<JsonObject> SendAsync(WebIqSettings settings, HttpMethod method, string path,
        JsonObject? body = null, bool allowMissing = false)
    {
        settings.Validate();
        string token;
        try { token = await getToken(); }
        catch (Exception error)
        {
            throw new InvalidOperationException(WebIqResults.Redact(error.Message, settings.WebIqKey));
        }
        using var request = new HttpRequestMessage(method, $"{settings.SearchEndpoint}/{path}?api-version={ApiVersion}");
        request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", token);
        if (body is not null)
            request.Content = new StringContent(body.ToJsonString(), Encoding.UTF8, "application/json");
        try
        {
            using var response = await client.SendAsync(request);
            string text = await response.Content.ReadAsStringAsync();
            int status = (int)response.StatusCode;
            string Safe(string value) => WebIqResults.Redact(value, settings.WebIqKey, token);
            Console.WriteLine($"HTTP status: {status}");
            Console.WriteLine("Request ID: " + Safe(response.Headers.TryGetValues("x-ms-request-id", out var ids)
                ? string.Join(", ", ids) : response.Headers.TryGetValues("request-id", out ids) ? string.Join(", ", ids) : ""));
            if (allowMissing && status == 404) return new JsonObject();
            if (!response.IsSuccessStatusCode)
            {
                Console.Error.WriteLine($"HTTP {status}: Search request failed. UTC (Date): {response.Headers.Date:R}");
                foreach (var header in response.Headers)
                    if (header.Key.Contains("request", StringComparison.OrdinalIgnoreCase) ||
                        header.Key.Contains("correlation", StringComparison.OrdinalIgnoreCase) ||
                        header.Key.Contains("trace", StringComparison.OrdinalIgnoreCase) ||
                        header.Key.Equals("retry-after", StringComparison.OrdinalIgnoreCase))
                        Console.Error.WriteLine(header.Key + ": " + Safe(string.Join(", ", header.Value)));
                string safe = Safe(text);
                Console.Error.WriteLine("Response body: " + safe[..Math.Min(8000, safe.Length)]);
                throw new HttpRequestException($"HTTP {status}: Search request failed; inspect redacted diagnostics above.", null, response.StatusCode);
            }
            JsonObject result;
            try
            {
                result = string.IsNullOrWhiteSpace(text) && method != HttpMethod.Post ? new JsonObject()
                    : JsonNode.Parse(Safe(text))?.AsObject() ?? throw new JsonException();
            }
            catch (Exception error) when (error is JsonException or InvalidOperationException)
            {
                throw new InvalidOperationException("Search returned invalid JSON; no complete result available.");
            }
            if (method == HttpMethod.Post) WebIqResults.Print(result, status, settings.WebIqKey, token);
            return result;
        }
        catch (HttpRequestException error)
        {
            throw new HttpRequestException(WebIqResults.Redact(error.Message, settings.WebIqKey, token), null, error.StatusCode);
        }
        catch (OperationCanceledException)
        {
            throw new TimeoutException("Search request timed out before a complete response; inspect network/service health. No automatic retry.");
        }
    }

    public async Task ProbeAsync(WebIqSettings settings)
    {
        settings.Validate();
        Console.WriteLine("Optional unauthenticated HTTPS probe; HTTP 401 is expected.");
        using var request = new HttpRequestMessage(HttpMethod.Get,
            $"{settings.SearchEndpoint}/knowledgebases/{settings.BaseName}?api-version={ApiVersion}");
        using var timeout = new CancellationTokenSource(TimeSpan.FromSeconds(20));
        try
        {
            using var response = await client.SendAsync(request, timeout.Token);
            Console.WriteLine("HTTPS completed; HTTP status: " + (int)response.StatusCode);
            Console.WriteLine("An HTTP response does not validate Search permissions, knowledge-base existence or Web IQ readiness.");
        }
        catch (Exception error) when (error is HttpRequestException or OperationCanceledException)
        {
            throw new InvalidOperationException("HTTPS check failed: " + error.GetType().Name +
                ". No usable HTTP response; network/TLS failure is not resolved in this process.");
        }
    }

    public static async Task<string> AzureCliTokenAsync(string tenant)
    {
        if (!Guid.TryParseExact(tenant, "D", out _)) throw new InvalidOperationException("Invalid tenant GUID.");
        var start = new ProcessStartInfo
        {
            FileName = OperatingSystem.IsWindows() ? "cmd.exe" : "az",
            UseShellExecute = false,
            RedirectStandardOutput = true,
            RedirectStandardError = true,
            CreateNoWindow = true
        };
        if (OperatingSystem.IsWindows())
            foreach (string argument in new[] { "/d", "/c", "az" }) start.ArgumentList.Add(argument);
        foreach (string argument in new[] { "account", "get-access-token", "--scope", Scope, "--tenant", tenant, "--output", "json" })
            start.ArgumentList.Add(argument);
        using var process = Process.Start(start) ?? throw new InvalidOperationException("Could not start Azure CLI.");
        Task<string> output = process.StandardOutput.ReadToEndAsync();
        Task<string> errors = process.StandardError.ReadToEndAsync();
        using var timeout = new CancellationTokenSource(TimeSpan.FromSeconds(60));
        try { await process.WaitForExitAsync(timeout.Token); }
        catch (OperationCanceledException)
        {
            if (!process.HasExited) process.Kill(entireProcessTree: true);
            await process.WaitForExitAsync();
            await Task.WhenAll(output, errors);
            throw new InvalidOperationException("Azure CLI authentication timed out before Search/Web IQ. Run az login --tenant AZURE_TENANT_ID.");
        }
        await errors;
        if (process.ExitCode != 0)
            throw new InvalidOperationException("Azure CLI authentication failed before Search/Web IQ. Run az login --tenant AZURE_TENANT_ID.");
        try
        {
            return JsonNode.Parse(await output)?["accessToken"]?.GetValue<string>() is { Length: > 0 } token
                ? token : throw new JsonException();
        }
        catch (JsonException)
        {
            throw new InvalidOperationException("Azure CLI returned no usable access token.");
        }
    }
}

public static class WebIqResults
{
    public static string Redact(string text, params string[] secrets)
    {
        foreach (string secret in secrets.Where(secret => !string.IsNullOrEmpty(secret)))
        {
            text = text.Replace(secret, "<redacted>", StringComparison.Ordinal);
            string escaped = JsonSerializer.Serialize(secret)[1..^1];
            text = text.Replace(escaped, "<redacted>", StringComparison.Ordinal);
            text = text.Replace(Uri.EscapeDataString(secret), "<redacted>", StringComparison.Ordinal);
        }
        return text;
    }

    public static void ValidateQuery(string question, string outputMode)
    {
        if (string.IsNullOrWhiteSpace(question)) throw new ArgumentException("Use a non-empty question.");
        if (outputMode is not ("answerSynthesis" or "extractiveData"))
            throw new ArgumentException("Use answerSynthesis or extractiveData.");
    }

    public static void Print(JsonObject result, int status, params string[] secrets)
    {
        void Write(string value) => Console.WriteLine(Redact(value, secrets));
        Console.WriteLine("\nActivity:");
        Write((result["activity"] ?? new JsonArray()).ToJsonString(new JsonSerializerOptions { WriteIndented = true }));
        Console.WriteLine("\nAnswer or extractive output:");
        foreach (JsonNode? message in result["response"] as JsonArray ?? new JsonArray())
            foreach (JsonNode? part in message?["content"] as JsonArray ?? new JsonArray())
                Write(part?["text"]?.ToString() ?? "");
        Console.WriteLine("\nReferences and retrieved passages:");
        foreach (JsonNode? reference in result["references"] as JsonArray ?? new JsonArray())
        {
            var data = reference?["sourceData"]?.DeepClone() as JsonObject ?? new JsonObject();
            try
            {
                if (JsonNode.Parse(data["content"]?.ToString() ?? "{}") is JsonObject parsed)
                    foreach (var property in parsed) data[property.Key] = property.Value?.DeepClone();
            }
            catch (JsonException) { }
            Write($"- {reference?["id"]} {data["title"]} {data["url"]}");
            string content = data["content"]?.ToString() ?? "";
            string safeContent = Redact(content, secrets);
            Write(safeContent[..Math.Min(2000, safeContent.Length)]);
        }
        if (status == 206 || result["activity"] is JsonArray activity &&
            activity.Any(step => step?["error"] is JsonNode error && error.ToJsonString() is not ("null" or "{}" or "\"\"" or "false")))
            throw new InvalidOperationException("Retrieval was partial or Web IQ failed. Inspect the activity above.");
        if (result["references"] is not JsonArray references || references.Count == 0)
            Console.WriteLine("No references returned. Inspect activity; this alone does not establish an authentication failure.");
    }
}


WebIqSettings settings = WebIqSettings.Load(Path.GetFullPath(".."));
HttpClient client = new HttpClient(new HttpClientHandler { AllowAutoRedirect = false })
{
    Timeout = TimeSpan.FromSeconds(120)
};
Func<Task<string>> tokenProvider = () => WebIqTransport.AzureCliTokenAsync(settings.Tenant);
WebIqTransport transport = new WebIqTransport(client, settings.Tenant, () => tokenProvider());
Console.WriteLine("Search: " + settings.SearchEndpoint);
Console.WriteLine("Web IQ source: " + settings.SourceName);
Console.WriteLine("Knowledge base: " + settings.BaseName);


### Optional troubleshooting only — HTTP 401 is expected

Skip the next cell in normal use (disabled by default). For repeated connection reset/TLS
errors, enable it **in the same failing kernel**. This unauthenticated read-only HTTPS probe
sends no Authorization or API-key headers, does not acquire a token and never invokes Web IQ.
HTTP 401 is expected: it confirms an HTTP response, **not** Search permissions, knowledge-base
existence or Web IQ readiness. Any other HTTP response also confirms a response, not
authenticated retrieval. A TLS/connection error means no usable response.

A successful terminal test does not establish connectivity in this kernel. Do not disable
certificate verification or bypass organizational proxy/network controls. Local proxy flags
below print only presence, never values/credentials. The probe cannot identify which network
component caused a reset.


In [ ]:
bool CHECK_CONNECTIVITY = false;
if (CHECK_CONNECTIVITY)
{
    Console.WriteLine($"UTC: {DateTime.UtcNow:O}\n.NET: {Environment.Version}\nSearch: {settings.SearchEndpoint}");
    Console.WriteLine("Proxy environment configured: " + new[] { "HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY" }
        .Any(name => !string.IsNullOrEmpty(Environment.GetEnvironmentVariable(name))));
    await transport.ProbeAsync(settings);
}
else Console.WriteLine("Optional unauthenticated connectivity probe skipped.");


## 2. Create the Web IQ source

Running this cell stores `WEBIQ_API_KEY` in the new source's `storedHeaders` authentication. It does not change sources used by notebooks 11 or 12. Rerunning updates this source, including its stored key.

Only the basic `web` tool is configured. `outputParsing: auto` is Search's response parser, not the Web IQ Auto tool. Passage-format guidance avoids full-document formats that the workshop profile previously rejected. This is planner guidance, not a server-side parameter override.

In [ ]:
JsonObject webiqSource = new JsonObject()
    {
        ["name"] = settings.SourceName,
        ["kind"] = "mcpServer",
        ["description"] = "Basic Web IQ web search for public information. For every web call, explicitly set contentFormat to passage; use query-dependent passages, not text, html or markdown.",
        ["resultsProcessing"] = "rerank",
        ["mcpServerParameters"] = new JsonObject
        {
            ["serverURL"] = "https://api.microsoft.ai/v3/mcp",
            ["authentication"] = new JsonObject
            {
                ["kind"] = "storedHeaders",
                ["storedHeadersParameters"] = new JsonObject
                {
                    ["headers"] = new JsonObject { ["x-apikey"] = settings.WebIqKey }
                }
            },
            ["tools"] = new JsonArray(new JsonObject
            {
                ["name"] = "web",
                ["outputParsing"] = new JsonObject { ["kind"] = "auto" },
                ["maxOutputTokens"] = 4000
            })
        }
    };
await transport.SendAsync(settings, HttpMethod.Put, $"knowledgesources/{settings.SourceName}", webiqSource);
Console.WriteLine($"Web IQ source ready: {settings.SourceName} | authentication: storedHeaders | tool: web");


## 3. Create the Web IQ-only knowledge base

This knowledge base references only the source above. The existing chat model plans the web query and synthesizes the answer. No indexes, ingestion or other knowledge sources are involved.

In [ ]:
JsonObject knowledgeBase = new JsonObject()
    {
        ["name"] = settings.BaseName,
        ["description"] = "Standalone public web retrieval through basic Web IQ search only.",
        ["retrievalInstructions"] = $"Use {settings.SourceName} to retrieve public web evidence for the question. For every web call explicitly set contentFormat to passage. Use specific search terms, locations and dates from the question. For current software releases prefer the official release or downloads page. For weather, search for the requested location and forecast date.",
        ["answerInstructions"] = "Write a concise final answer from the retrieved passages and cite supporting reference IDs. Treat MCP web passages as evidence already retrieved for this request. If evidence does not establish the requested fact, identify the missing information. Distinguish forecast dates from page crawl dates and stable releases from prereleases. Do not infer the latest release from an incomplete list of older versions.",
        ["outputMode"] = "answerSynthesis",
        ["knowledgeSources"] = new JsonArray(new JsonObject { ["name"] = settings.SourceName }),
        ["models"] = new JsonArray(new JsonObject
        {
            ["kind"] = "azureOpenAI",
            ["azureOpenAIParameters"] = new JsonObject
            {
                ["resourceUri"] = settings.ModelEndpoint,
                ["deploymentId"] = settings.Deployment,
                ["modelName"] = settings.Model
            }
        }),
        ["retrievalReasoningEffort"] = new JsonObject { ["kind"] = "low" }
    };
await transport.SendAsync(settings, HttpMethod.Put, $"knowledgebases/{settings.BaseName}", knowledgeBase);
Console.WriteLine("Web IQ-only knowledge base ready: " + settings.BaseName);


## 4. Choose a question and output mode

The default is an explicitly located, dated weather question. Replace it with another public question, such as `What is the latest stable Python release and its release date, according to Python.org?`.

Start with `answerSynthesis`. Then change `output_mode` to `extractiveData` and rerun this cell and the next cell to inspect retrieval without a generated answer. The two modes make separate searches; their results can differ. This does not change the stored knowledge-base configuration.

Web search results are not a live sensor or clock. Check that retrieved passages actually support the requested place, date and fact rather than assuming a returned reference proves answer accuracy.

In [ ]:
string today = DateTime.UtcNow.ToString("yyyy-MM-dd");
string question = $"What is the weather forecast for Seattle, Washington, on {today}? Cite the forecast source and date.";
string outputMode = "answerSynthesis";
WebIqResults.ValidateQuery(question, outputMode);
Console.WriteLine(WebIqResults.Redact($"Question: {question}\nOutput mode: {outputMode}", settings.WebIqKey));


In [ ]:
WebIqResults.ValidateQuery(question, outputMode);
JsonObject retrieveRequest = new JsonObject
        {
            ["messages"] = new JsonArray(new JsonObject
            {
                ["role"] = "user",
                ["content"] = new JsonArray(new JsonObject { ["type"] = "text", ["text"] = question })
            }),
            ["outputMode"] = outputMode,
            ["includeActivity"] = true,
            ["maxRuntimeInSeconds"] = 60,
            ["knowledgeSourceParams"] = new JsonArray(new JsonObject
            {
                ["knowledgeSourceName"] = settings.SourceName,
                ["kind"] = "mcpServer",
                ["includeReferences"] = true,
                ["includeReferenceSourceData"] = true
            })
        };
JsonObject result = await transport.SendAsync(settings, HttpMethod.Post,
    $"knowledgebases/{settings.BaseName}/retrieve", retrieveRequest);
// SendAsync prints redacted activity, text, URLs/passages and rejects partial/source errors.


## 5. Interpret the result and try the portal

| Evidence | Interpretation |
|---|---|
| HTTP 401/403 from Search | Inspect Search sign-in, tenant, RBAC and network access; this is not automatically a Web IQ error. |
| MCP activity reports `auth_missing_token` | Web IQ did not receive accepted authentication. This source uses a stored `x-apikey`, not APIM header injection. Verify the key/profile and rerun source creation after correcting it. |
| MCP activity reports another error | Inspect the tool arguments and actual error. For a content-format rejection, check that `contentFormat` is `passage`. |
| MCP returned relevant passages, but the answer says it cannot search | The answer contradicts retrieved evidence. Inspect the passages and compare `extractiveData`; do not diagnose a missing key from answer wording alone. |
| MCP returned no relevant evidence | Check the search terms, location and date. HTTP 200 or a result count does not prove the question was answered. |

In the Foundry IQ portal, select this Search service and the printed `WORKSHOP_PREFIX`-`nb14-kb` knowledge base. When selecting a source for another knowledge base, use this notebook's `WORKSHOP_PREFIX`-`nb14-webiq` source, not the keyless `iq-workshop-webiq` APIM source.

The key belongs to the knowledge source, not to a knowledge base or chat. It is shared by authorized callers. Rotate it in Web IQ and rerun source creation to update the stored value; revoke it when no longer needed. Do not print `webiq_source`, which contains the key.

### Web IQ-only knowledge base in the portal

![Web IQ-only knowledge base in the Foundry IQ portal, with one active Web IQ source and a cited Seattle weather answer](../docs/webiq-only-knowledge-base-portal.png)

The left pane shows the single active Web IQ knowledge source. The right pane shows a Seattle weather question, an answer and a source reference. This notebook enables only the `web` tool; it does not configure a separate `news` tool.

The screenshot is illustrative: its resource prefix, date and weather answer will differ from your run. The `nb14` resource suffix is retained from this lab's original numbering even though the notebook is now 09. Inspect the cited passages to check the place, date and forecast; a generated answer or citation alone does not prove accuracy. Sending another question in the portal makes a separate live retrieval request.

## 6. Optional cleanup

Leave cleanup disabled while testing in the portal. Enabling it deletes only this notebook's knowledge base and source, in that order. It does not remove the shared Search service, model deployment, Web IQ profile/key, or resources from notebooks 11 and 12.

In [ ]:
bool CLEAN_UP = false;
if (CLEAN_UP)
{
    foreach (var (collection, name) in new[]
    {
        ("knowledgebases", settings.BaseName), ("knowledgesources", settings.SourceName)
    })
    {
        await transport.SendAsync(settings, HttpMethod.Delete, $"{collection}/{name}", allowMissing: true);
        Console.WriteLine("Removed or already absent: " + name);
    }
}
else Console.WriteLine("Cleanup disabled; Web IQ-only resources remain available.");


## Release local resources

After your last request, dispose the HTTP client. No Azure resource is deleted.
Rerun Configuration to create a new client before more requests. Each CLI credential
acquisition disposes its process and timeout; no persistent credential is retained.


In [ ]:
client.Dispose();

## References

- [MCP server knowledge sources and authentication](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-mcp-server)
- [Create a knowledge base](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-create-knowledge-base)
- [Retrieve action and response](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve)
- [Answer synthesis and per-request output mode](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-answer-synthesis)